In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pygmm

import pyrvt

In [2]:
import os

from shared import (  # type: ignore
    compute_covariance,
    process_fas_for_scenario,
    process_realizations,
)

meanInSquares = [False, True, False, True]
GMM_models = ["BA19", "BA19", "Stafford2020", "Stafford2020"]
corr_models = ["BA18", "BA18", "Stafford", "Stafford"]
v_s30 = 400
# Define parameters for the sensitivity analysis
mags = [5.0, 6.0, 7.0, 8.0]
dists = [5, 30, 100]
freqs_osc = np.geomspace(0.2, 100, 20)


# Set the save path to the created folder
def figure_address(filename):
    return os.path.join(folder_name, filename)


for i in range(len(meanInSquares)):
    meanInSquare = meanInSquares[i]
    GMM_model = GMM_models[i]
    corr_model = corr_models[i]
    if meanInSquare:
        print("Mean in square")
    else:
        print("Mean in natural")

    # Create folder based on meanInSquare setting
    folder_name = "meanInSquare" if meanInSquare else "meanInNat"
    folder_name = folder_name + GMM_model
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    for mag in mags:
        for dist in dists:
            print(f"Processing: Magnitude {mag}, Distance {dist} km")
            s = pygmm.model.Scenario(
                mag=mag,
                dist_jb=dist,
                dist_x=dist,
                dist_rup=dist,
                dip=90,
                v_s30=v_s30,
                depth_tor=0,
                mechanism="SS",
            )
            freqs, fas, std = process_fas_for_scenario(
                s, model=GMM_model, interpolate=False
            )
            if corr_model == "BA18":
                cov = compute_covariance(freqs, corr_model, std=std)
            elif corr_model == "Stafford2020":
                cov = compute_covariance(freqs, corr_model, magnitude=mag)

            # Get duration from Afshari & Stewart
            as_model = pygmm.AfshariStewart2016(s)
            duration = as_model.duration.D_5t75
            ln_eas = np.log(fas)
            n_realizations = 1000
            # With correlation
            (
                realizations_corr,
                psa_realizations_corr,
                median_psa_corr,
                mean_psa_corr,
                sd,
            ) = process_realizations(
                cov,
                n_realizations,
                ln_eas,
                freqs,
                freqs_osc,
                duration,
                meanInSquare=meanInSquare,
                figure_address=figure_address(f"{mag} {dist} cor"),
            )

            # Without correlation (diagonal covariance matrix)
            cov_uncorr = np.diag(
                np.diag(cov)
            )  # Only keep diagonal elements, zero out off-diagonal
            (
                realizations_uncorr,
                psa_realizations_uncorr,
                median_psa_uncorr,
                mean_psa_uncorr,
                _,
            ) = process_realizations(
                cov_uncorr,
                n_realizations,
                ln_eas,
                freqs,
                freqs_osc,
                duration,
                meanInSquare=meanInSquare,
                figure_address=figure_address(f"{mag} {dist} uncor"),
            )
            plt.figure(figsize=(8, 5))
            mot_orig = pyrvt.motions.RvtMotion(
                freqs,
                fas,
                duration,
                peak_calculator=pyrvt.peak_calculators.Vanmarcke1975(
                    use_nonstationarity_factor=False
                ),
            )
            psa_orig = mot_orig.calc_osc_accels(freqs_osc, 0.05)
            plt.loglog(
                1.0 / freqs_osc,
                psa_orig,
                "k-",
                label="Single analysis median FAS to PSA",
            )
            # Plot the mean PSAs
            if meanInSquare:
                # Process the original FAS to PSA using V75
                mot_orig = pyrvt.motions.RvtMotion(
                    freqs,
                    fas * np.exp(std**2),
                    duration,
                    peak_calculator=pyrvt.peak_calculators.Vanmarcke1975(
                        use_nonstationarity_factor=False
                    ),
                )
                psa_orig = mot_orig.calc_osc_accels(freqs_osc, 0.05)
                # Create a plot to compare results
                plt.loglog(
                    1.0 / freqs_osc,
                    psa_orig,
                    "k--",
                    label=r"$\sqrt{Mean\  FAS^2} to PSA$",
                )
                plt.loglog(
                    1.0 / freqs_osc,
                    psa_orig * np.exp(-(sd**2)),
                    "r--",
                    label=r"$\sqrt{Mean\  FAS^2} to PSA corrected$",
                )

                # Process the original FAS to PSA using V75
                mot_orig = pyrvt.motions.RvtMotion(
                    freqs,
                    fas,
                    duration,
                    peak_calculator=pyrvt.peak_calculators.Vanmarcke1975(
                        use_nonstationarity_factor=False
                    ),
                )
                psa_orig = mot_orig.calc_osc_accels(freqs_osc, 0.05)
                plt.loglog(
                    1.0 / freqs_osc,
                    mean_psa_corr,
                    "b--",
                    linewidth=2,
                    label=r"$\sqrt{Mean PSA^2} (with correlation)$",
                )

                plt.loglog(
                    1.0 / freqs_osc,
                    mean_psa_uncorr,
                    "g--",
                    linewidth=2,
                    label=r"$\sqrt{Mean PSA^2} (without correlation)$",
                )
            else:
                # Process the original FAS to PSA using V75
                mot_orig = pyrvt.motions.RvtMotion(
                    freqs,
                    fas * np.exp(std**2 / 2),
                    duration,
                    peak_calculator=pyrvt.peak_calculators.Vanmarcke1975(
                        use_nonstationarity_factor=False
                    ),
                )
                psa_orig = mot_orig.calc_osc_accels(freqs_osc, 0.05)
                # Create a plot to compare results
                plt.loglog(
                    1.0 / freqs_osc,
                    psa_orig,
                    "k--",
                    label="Single analysis mean FAS to PSA",
                )

                plt.loglog(
                    1.0 / freqs_osc,
                    mean_psa_corr,
                    "b--",
                    linewidth=2,
                    label="Mean PSA (with correlation)",
                )
                plt.loglog(
                    1.0 / freqs_osc,
                    mean_psa_uncorr,
                    "g--",
                    linewidth=2,
                    label="Mean PSA (without correlation)",
                )

            # Plot the mean PSAs
            plt.loglog(
                1.0 / freqs_osc,
                median_psa_corr,
                "b-",
                linewidth=2,
                label="Median PSA (with correlation)",
            )
            plt.loglog(
                1.0 / freqs_osc,
                median_psa_uncorr,
                "g-",
                linewidth=2,
                label="Median PSA (without correlation)",
            )

            plt.xlabel("Period (s)")
            plt.ylabel("PSA (g)")
            plt.grid(True, which="both", ls="-", alpha=0.2)
            plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
            plt.title(f"M{mag} Rrup{dist} comparison")
            plt.tight_layout()
            plt.savefig(figure_address(f"{mag} {dist} comparison_logspace.jpeg"))
            plt.close()
            mot_orig = pyrvt.motions.RvtMotion(
                freqs,
                fas,
                duration,
                peak_calculator=pyrvt.peak_calculators.Vanmarcke1975(
                    use_nonstationarity_factor=False
                ),
            )
            psa_orig = mot_orig.calc_osc_accels(freqs_osc, 0.05)
            # Calculate the ratios between mean PSAs and original PSA
            ratio_log_corr = np.log(median_psa_corr / psa_orig)
            ratio_log_uncorr = np.log(median_psa_uncorr / psa_orig)

            plt.figure(figsize=(8, 5))
            plt.semilogx(
                1.0 / freqs_osc,
                ratio_log_corr,
                "b-",
                linewidth=2,
                label="With correlation",
            )
            plt.semilogx(
                1.0 / freqs_osc,
                ratio_log_uncorr,
                "g-",
                linewidth=2,
                label="Without correlation",
            )
            plt.grid(True, which="both", ls="-", alpha=0.2)
            plt.xlabel("Period (s)")
            plt.ylabel("log of Ratio (Median PSA / Based on median FAS)")
            plt.ylim(0, 0.6)
            plt.title(f"M{mag} Rrup{dist} ratio")
            plt.legend()
            plt.tight_layout()
            plt.savefig(figure_address(f"{mag} {dist} ratio_logspace.jpeg"))
            plt.close()

Mean in natural
Processing: Magnitude 5.0, Distance 5 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 5.0, Distance 30 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 5.0, Distance 100 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 6.0, Distance 5 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 6.0, Distance 30 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 6.0, Distance 100 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 7.0, Distance 5 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 7.0, Distance 30 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 7.0, Distance 100 km
Using Bayless and Abrahamson 2018 model


Processing: Magnitude 8.0, Distance 5 km
Using Bayless and Abrahamson 2018 model


Processing: Magnitude 8.0, Distance 30 km
Using Bayless and Abrahamson 2018 model


Processing: Magnitude 8.0, Distance 100 km
Using Bayless and Abrahamson 2018 model
Mean in square
Processing: Magnitude 5.0, Distance 5 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 5.0, Distance 30 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 5.0, Distance 100 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 6.0, Distance 5 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 6.0, Distance 30 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 6.0, Distance 100 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 7.0, Distance 5 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 7.0, Distance 30 km
Using Bayless and Abrahamson 2018 model
Processing: Magnitude 7.0, Distance 100 km
Using Bayless and Abrahamson 2018 model


Processing: Magnitude 8.0, Distance 5 km
Using Bayless and Abrahamson 2018 model


Processing: Magnitude 8.0, Distance 30 km
Using Bayless and Abrahamson 2018 model


... (3 outputs left out)

Processing: Magnitude 8.0, Distance 100 km
Mean in square
Processing: Magnitude 5.0, Distance 5 km
Processing: Magnitude 5.0, Distance 30 km
Processing: Magnitude 5.0, Distance 100 km
Processing: Magnitude 6.0, Distance 5 km
Processing: Magnitude 6.0, Distance 30 km
Processing: Magnitude 6.0, Distance 100 km
Processing: Magnitude 7.0, Distance 5 km
Processing: Magnitude 7.0, Distance 30 km
Processing: Magnitude 7.0, Distance 100 km


Processing: Magnitude 8.0, Distance 5 km


Processing: Magnitude 8.0, Distance 30 km


Processing: Magnitude 8.0, Distance 100 km


In [3]:
cov[:3, :3]

array([[0.49566266, 0.13026288, 0.12918446],
       [0.13026288, 0.49566266, 0.48871043],
       [0.12918446, 0.48871043, 0.49932518]])

In [4]:
freqs[1] / freqs[0]

12.048192771084338